# Men's Tournament Scoring

Using the model generated brackets, find the brackets that best perform against a hypothetical public pool. 

In [1]:
import json

with open(f'../data/meta/mens/metadata.json', "r") as file:
    json_data = json.load(file)

SEASON = json_data['SEASON']

previously_used_brackets_filepaths = json_data['season_data'][str(SEASON)]['previously_used_brackets_filepaths'].copy()

SEASON, previously_used_brackets_filepaths

(2025,
 ['../data/brackets/mens/2025_03_23/bracket_0.pkl',
  '../data/brackets/mens/2025_03_23/bracket_1.pkl',
  '../data/brackets/mens/2025_03_23/bracket_2.pkl',
  '../data/brackets/mens/2025_03_24\\bracket_0.pkl',
  '../data/brackets/mens/2025_03_24\\bracket_1.pkl',
  '../data/brackets/mens/2025_03_24\\bracket_2.pkl',
  '../data/brackets/mens/2025_03_22\\bracket_0.pkl',
  '../data/brackets/mens/2025_03_22\\bracket_1.pkl',
  '../data/brackets/mens/2025_03_22\\bracket_2.pkl'])

### Setup Data

In [2]:
import pickle

with open(f'../data/simulations/mens/mens_simulation_{SEASON}.pkl', 'rb') as f:
    simulation_data = pickle.load(f)

result_brackets = simulation_data['result_brackets']
candidate_brackets = simulation_data['candidate_brackets']
public_brackets = simulation_data['public_brackets']

del simulation_data

used_brackets = []
for filepath in previously_used_brackets_filepaths:
    with open(filepath, 'rb') as f:
        used_bracket = pickle.load(f)

    used_brackets.append(used_bracket)

len(result_brackets), len(candidate_brackets), len(public_brackets), len(used_brackets)

(30000, 40000, 20000, 9)

### Score Brackets

In [3]:
import numpy as np

def dict_to_tuple(d):
    """Get values of dictionary in a tuple, sorted by key"""
    return tuple(d[key] for key in sorted(d.keys()))

result_array = np.array(tuple(dict_to_tuple(r.picks) for r in result_brackets))
candidate_array = np.array(tuple(dict_to_tuple(c.picks) for c in candidate_brackets))
public_array = np.array(tuple(dict_to_tuple(p.picks) for p in public_brackets))
used_array = np.array(tuple(dict_to_tuple(u.picks) for u in used_brackets))

result_array.shape, candidate_array.shape, public_array.shape, used_array.shape

((30000, 63), (40000, 63), (20000, 63), (9, 63))

In [4]:
round_scores = np.array([10]*32 + [20]*16 + [40]*8 + [80]*4 + [160]*2 + [320]*1)

def score_brackets(brackets, ground_truth):
    """Score array of brackets compared to a bracket treated as ground truth"""
    return np.sum((brackets == ground_truth)*round_scores, axis=1)

In [5]:
from tqdm.autonotebook import tqdm
from joblib import Parallel, delayed

# each row is a public bracket and each column is the tournament simulation

public_scores = (
    np.array(Parallel(n_jobs=-1)(delayed(score_brackets)(public_array, result) for result in tqdm(result_array)))
    .transpose()
)

public_scores.shape

C:\Users\mhugh\AppData\Local\Temp\ipykernel_16780\1276773675.py:1: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


  0%|          | 0/30000 [00:00<?, ?it/s]

(20000, 30000)

In [6]:
# each row is a candidate bracket and each column is the tournament simulation

candidate_scores = (
    np.array(Parallel(n_jobs=-1)(delayed(score_brackets)(candidate_array, result) for result in tqdm(result_array)))
    .transpose()
)

candidate_scores.shape

  0%|          | 0/30000 [00:00<?, ?it/s]

(40000, 30000)

In [7]:
# each row is a candidate bracket and each column is the tournament simulation

if used_array.shape[0] != 0:
    used_scores = (
        np.array(Parallel(n_jobs=-1)(delayed(score_brackets)(used_array, result) for result in tqdm(result_array)))
        .transpose()
    )
else:
    used_scores = None

if isinstance(used_scores, np.ndarray):
    display(used_scores.shape)
else:
    display(type(used_scores))

  0%|          | 0/30000 [00:00<?, ?it/s]

(9, 30000)

### Display Brackets

In [8]:
def get_best_brackets(
    pool_size: int, 
    number_of_brackets: int, 
    first_payout: int, 
    second_payout: int, 
    third_payout: int,
):
    """
    Get the best n possible brackets for a given pool size. 
    Subsequent brackets choices will headge previous bracket choices. 
    """
    third_place = np.quantile(public_scores, q=1-3/pool_size, axis=0)
    second_place = np.quantile(public_scores, q=1-2/pool_size, axis=0)
    first_place = np.quantile(public_scores, q=1-1/pool_size, axis=0)

    candidate_prizes = (
        (candidate_scores > third_place)*(third_payout) + 
        (candidate_scores > second_place)*(second_payout - third_payout) + 
        (candidate_scores > first_place)*(first_payout - second_payout)
    )

    # account for previously used brackets
    if used_scores is not None:
        used_prizes = (
            (used_scores > third_place)*(third_payout) + 
            (used_scores > second_place)*(second_payout - third_payout) + 
            (used_scores > first_place)*(first_payout - second_payout)
        )

        # hedging: remove any ground truths that the previously used brackets won in
        ignore_indexes = np.where(used_prizes.max(axis=0) != 0)[0]
        candidate_prizes = np.delete(candidate_prizes, ignore_indexes, axis=1)

    best_brackets = []  # indexes of best brackets from candidates
    for _ in tqdm(range(number_of_brackets)):
        best_available_bracket = candidate_prizes.mean(axis=1).argmax()  # index of best available bracket given previous brackets selected
        prize = candidate_prizes.mean(axis=1)[best_available_bracket]
        best_brackets.append((best_available_bracket, prize))

        # hedging: remove any ground truths that the best available bracket won in
        ignore_indexes = np.where(candidate_prizes[best_available_bracket] != 0)[0]
        candidate_prizes = np.delete(candidate_prizes, ignore_indexes, axis=1)

    return best_brackets

In [9]:
bb = get_best_brackets(
    pool_size=10, 
    number_of_brackets=3, 
    first_payout=85, 
    second_payout=12, 
    third_payout=3,
)

bb

  0%|          | 0/3 [00:00<?, ?it/s]

C:\Users\mhugh\AppData\Local\Temp\ipykernel_16780\988938032.py:36: RuntimeWarning: Mean of empty slice.
  best_available_bracket = candidate_prizes.mean(axis=1).argmax()  # index of best available bracket given previous brackets selected
c:\Users\mhugh\anaconda3\envs\clean2\lib\site-packages\numpy\core\_methods.py:121: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
C:\Users\mhugh\AppData\Local\Temp\ipykernel_16780\988938032.py:37: RuntimeWarning: Mean of empty slice.
  prize = candidate_prizes.mean(axis=1)[best_available_bracket]


[(3320, 85.0), (0, nan), (0, nan)]

### Display Brackets

In [10]:
import os
from typing_extensions import List, Tuple

def display_brackets(bb: List[Tuple], save_to_folder: str = None):
    # make directory if it does not exist yet
    if save_to_folder is not None:
        os.makedirs(save_to_folder, exist_ok=True)

    for index, (b, _) in enumerate(bb):
        bracket = candidate_brackets[b]

        print(f'BRACKET {index}\n')
        print(bracket)

        if save_to_folder is not None:
            filepath = os.path.join(save_to_folder, f'bracket_{index}.pkl')
            with open(filepath, 'wb') as f:
                pickle.dump(bracket, f)

            json_data['season_data'][str(SEASON)]['previously_used_brackets_filepaths'].append(filepath)

    if save_to_folder is not None:
        with open(f'../data/meta/mens/metadata.json', "w") as f:
            json.dump(json_data, f, indent=4)  # indent=4 makes it more readable

In [11]:
display_brackets(bb, save_to_folder=None)

BRACKET 0

------------------------------

ROUND OF 64 WINNERS

REGION W
R1W1: Duke (1)
R1W8: Baylor (9)
R1W5: Oregon (5)
R1W4: Arizona (4)
R1W6: BYU (6)
R1W3: Wisconsin (3)
R1W7: Vanderbilt (10)
R1W2: Alabama (2)

REGION X
R1X1: Houston (1)
R1X8: Gonzaga (8)
R1X5: Clemson (5)
R1X4: Purdue (4)
R1X6: Xavier (11)
R1X3: Kentucky (3)
R1X7: UCLA (7)
R1X2: Tennessee (2)

REGION Y
R1Y1: Auburn (1)
R1Y8: Creighton (9)
R1Y5: UC San Diego (12)
R1Y4: Yale (13)
R1Y6: Mississippi (6)
R1Y3: Iowa St (3)
R1Y7: New Mexico (10)
R1Y2: Michigan St (2)

REGION Z
R1Z1: Norfolk St (16)
R1Z8: Connecticut (8)
R1Z5: Memphis (5)
R1Z4: Maryland (4)
R1Z6: Missouri (6)
R1Z3: Texas Tech (3)
R1Z7: Kansas (7)
R1Z2: St John's (2)

------------------------------

ROUND OF 32 WINNERS

REGION W
R2W1: Duke (1)
R2W4: Arizona (4)
R2W3: Wisconsin (3)
R2W2: Alabama (2)

REGION X
R2X1: Houston (1)
R2X4: Purdue (4)
R2X3: Kentucky (3)
R2X2: Tennessee (2)

REGION Y
R2Y1: Creighton (9)
R2Y4: Yale (13)
R2Y3: Iowa St (3)
R2Y2: New Me

In [12]:
# display_brackets(bb, save_to_folder='../data/brackets/mens/2025_03_22')